# FigS2 

In [ ]:
DoTypeDimPlot <- function(object,vars,clusters,label.size,title = NULL,legend = NULL,arrow.num = 5,ncol = 4,
                          tissue = NULL){
        data <- data.frame(x = 0,y =0)
        p.arrow <- ggplot(data = data,aes(x = x,y = y)) + geom_point(size=1, alpha=1) + 
            geom_segment(aes(,x = x , y = y , xend = x +4, yend = y ), colour = "black",
                     size=0.5,arrow = arrow(length = unit(0.5,"cm")))+ 
            geom_segment(aes(x = x  , y = y , xend = x , yend =y + 4), colour = "black", 
                     size=0.5,arrow = arrow(length = unit(0.5,"cm"))) +
            annotate("text", x = data$x+2, y =data$y-1, label = "tSNE_1", color="black",
                 size = 4,fontface="bold" ) + 
            annotate("text", x = data$x-1, y =data$y + 2, label = "tSNE_2",color="black",
                 size = 4, fontface="bold" ,angle=90) + theme_bw()+
            theme(plot.margin=unit(c(0.1, 0.1, 0.1, 0.1), "inches"),legend.title=element_blank()) +
                theme(panel.grid.major = element_blank(), #主网格线
                panel.grid.minor = element_blank(), #次网格线
                panel.border = element_blank(), #边框
                axis.title = element_blank(),  #轴标题
                axis.text = element_blank(), # 文本
                axis.ticks = element_blank())
        plot.data <- FetchData(object,vars = vars)
        #plot.data$clusters <- as.character(plot.data$clusters) # !!!
        #plot.data <- plot.data %>% mutate(celltype2 = paste(clusters,celltypes,sep = " "))
        #plot.data$celltype2 <- factor(plot.data$celltype2,levels = clusters$celltype)
        
        if(is.null(tissue)){
            tissue <- unique(object@meta.data$tissue2)
        }
        print(tissue)
        plotlist <- list()
        for(i in 1:length(tissue)){
            print(i)
            type <- tissue[i]
            print(type)
            plot.data$show <- "other"
            plot.data$show[plot.data$tissue2 == type] <- type
            print(table(plot.data$show))
            # num #
            num <- format(length(plot.data$tissue2[plot.data$tissue2 == type]),big.mark = ",",trim = TRUE)
            title <- paste(type," (",num," cells)",sep = "")
            ### ###
            plot.data$clusters[plot.data$tissue2 != type] <- NA
            centers <- plot.data %>% group_by(clusters) %>% summarise(tSNE_1 = median(tSNE_1),tSNE_2 = median(tSNE_2)) %>%
                as.data.frame()   
            #cols <- c("grey","red")
            cols <- c('#342e6c','#d7eb28')
            names(cols) <- c("other",type)
            print(cols)
            p <- ggplot(data=plot.data,aes(x=tSNE_1, y=tSNE_2))+
                geom_point(alpha=0.2, size=0.0000000001,aes(colour=show)) +
                guides(colour=guide_legend(override.aes=list(size=5,alpha = 1)))+
                scale_color_manual(values=cols)+
                theme_bw()+
                labs(x= "tSNE_1",y="tSNE_2",title= title)+
                theme(plot.margin=unit(c(0.1, 0.1, 0.1, 0.1), "inches"),legend.title=element_blank()) +
                       theme(panel.grid.major = element_blank(), #主网格线
                       panel.grid.minor = element_blank(), #次网格线
                       panel.border = element_blank(), #边框
                       axis.title = element_blank(),  #轴标题
                       axis.text = element_blank(), # 文本
                       axis.ticks = element_blank(),plot.title = element_text(size =30,hjust = 0.5,
                                                                          face = "bold")) +
                theme(legend.text=element_text(colour="black",size=12),legend.position="right",
                      plot.title=element_text(hjust=0.5))+
                geom_point(data=centers, mapping=aes(x=tSNE_1, y=tSNE_2), size=0, alpha=0) +
                geom_text(data=centers, mapping=aes(label=clusters), size=label.size, fontface="plain")
           if(i == arrow.num){
            p <- p +  inset_element(p.arrow,left = 0,bottom = 0,right = 0.2,top = 0.2,align_to= "full",on_top = TRUE)
            } 
            if(is.null(legend)){
                plotlist[[i]] <- p & NoLegend()
            } else {
                plotlist[[i]] <- p
            }
        }
        p.whole <-  wrap_plots(plotlist,ncol=ncol)
        return(p.whole)                                                        
}

In [ ]:
DoVlnPlot2 <- function(Object,title,title.position){
    Object@meta.data$Percent.MT <-  Object@meta.data$percent.mt
    p <- VlnPlot(Object,features = c("nCount_RNA","nFeature_RNA","Percent.MT"),group.by = "mice",pt.size = 0,ncol =3) &
        theme(plot.title = element_text(size = 24,family = "Times",face = "bold",hjust = 0.5),
              axis.text.x = element_text(size = 21,family = "Times",face = "bold",margin = margin(5,0,0,0)),
              axis.text.y = element_text(size = 21,family = "Times",face = "bold",margin = margin(0,5,0,0)),
              axis.title = element_blank(),
              axis.line = element_line(linetype = 1,color= "black",size = 1),
              axis.ticks  = element_line(color = "black",size = 1,lineend = 2))
    if(title.position == "top"){
        #p <- p + plot_annotation(title = title,theme=theme(plot.title = element_text(size  = 50,family = "Times")))+plot_layout(guides = "collect", 
        #    nrow = 1)  # 单个patchwork plot 
        pt <-  ggdraw() + draw_label(title,fontfamily = "Times",fontface = 'bold',hjust = 0.5,vjust = 0.5,size = 40) # 多个
        p <- pt / p + plot_layout(heights = c(1, 4))
    } else { # bottom
        p <- p /textGrob(title,just = "bottom",x = 0.55,y=0.3,gp = gpar(fontsize = 50))   + plot_layout(heights = c(4, 1))
    }
    #print(title.position)
    return(p)
}
DoQCPlot <-  function(filelist,title.position = "top"){
    plotlist <- list()
    for(i in (1:length(filelist$tissue))){
        print(filelist$tissue[i])
        print(filelist$path[i])
        RDS <- readRDS(filelist$path[i])
        print(RDS)
        plotlist[[filelist$tissue[i]]] <- DoVlnPlot2(Object = RDS,title = filelist$tissue[i],title.position = title.position)
        print("----  End ----")
        rm(RDS)
        gc()
    }
    return(plotlist)
}
date()

# FigS3

In [ ]:
AddArrowFeaturePlot <- function(plot,plot.num,arrow.num,ncol,title = NULL){
    data <- data.frame(x = 0,y =0)
    p.arrow <- ggplot(data = data,aes(x = x,y = y)) + geom_point(size=1, alpha=1) + 
        geom_segment(aes(,x = x , y = y , xend = x +4, yend = y ), colour = "black", size=0.5,
                     arrow = arrow(length = unit(0.5,"cm")))+ 
        geom_segment(aes(x = x  , y = y , xend = x , yend =y + 4), colour = "black", size=0.5,
                     arrow = arrow(length = unit(0.5,"cm"))) +
        annotate("text", x = data$x+2, y =data$y-1, label = "tSNE_1", color="black",size = 4,fontface="bold" ) + 
        annotate("text", x = data$x-1, y =data$y + 2, label = "tSNE_2",color="black",size = 4, 
                 fontface="bold" ,angle=90) + theme_bw()+
        theme(plot.margin=unit(c(0.1, 0.1, 0.1, 0.1), "inches"),legend.title=element_blank()) +
               theme(panel.grid.major = element_blank(), #主网格线
               panel.grid.minor = element_blank(), #次网格线
               panel.border = element_blank(), #边框
               axis.title = element_blank(),  #轴标题
               axis.text = element_blank(), # 文本
               axis.ticks = element_blank())
    if(!is.null(title)){  
        for(i in 1:length(title)){
            print(i)
            print(title[i])
            plot[[i]]$labels$title <- title[i]
        }
        print("Title Change Finished!")
    }
    plotlist <- list()
    for(i in 1:plot.num){
        plotlist[[i]] <- plot[[i]]
        if(i == arrow.num){
            print(i)
            plotlist[[i]] <- plot[[i]] + inset_element(p.arrow,left = 0,bottom = 0,right = 0.2,
                                                     top = 0.2,align_to= "full",on_top = TRUE)
        }
    }
    p2 <- wrap_plots(plotlist,ncol=ncol) 
    p.out <- p2 & theme(plot.title = element_text(size = 25,face = "bold",hjust = 0.5),
              axis.text.x = element_blank(),
              axis.text.y = element_blank(),
              axis.title.x = element_blank(),
              axis.title.y = element_blank(),
              axis.line = element_blank(),
               axis.ticks = element_blank(),
              legend.text = element_text(size = 20,face = "bold"),
              legend.title = element_text(size = 25,face = "bold"))
    return(p.out)
}
date()

# FigS4 

In [ ]:
SmoothLinePlot <- function(data,label,annot_x,annot_y,y_title,color = c("#F8766D","#00BFC4")){
  theme_self <- theme(
    axis.text.y = element_text(size = 10,face = "bold",margin  = margin(0,5,0,0)),
    axis.text.x = element_text(size = 10,face = "bold",margin  = margin(5,0,0,0)),
    axis.title.x= element_text(size = 12,face = "bold",margin  = margin(0,0,0,0)),
    axis.title.y= element_text(size = 12,face = "bold",margin  = margin(0,0,0,0)),
    axis.line = element_line(linetype = 1,color= "black",size = 1),
    legend.text = element_text(size = 8,face = "bold"),
    legend.title = element_text(size = 10,face = "bold"),
    panel.grid = element_blank(),
    plot.margin =  unit(c(0.5,0.5,0.5,0.5),"cm"), # lwm 2023 -7 - 7 
    axis.ticks  = element_line(color = "black",size = 1,lineend = 2),
    plot.title = element_text(hjust = 0.5,size = 30,face = "bold"))
  p <- ggplot(data,aes(x = Layer,y = value,color = Mice,group = Mice)) + 
    geom_smooth(aes(color = Mice),method="lm", se = T,color = "black", # gray85
                fill = "gray70",formula = y ~ x,size = 0.5)  +  # size = 0.001 line geom_line()  + 
    geom_point(size =3.5,shape = 20)  + labs(y = y_title ) + #  ylim(4,10) + 
    scale_color_manual(values = c(GF = color[1],SPF = color[2] )) + 
    theme_classic() +theme_self + 
    theme(legend.position = c(1,1),legend.justification = c(1,1),
          legend.margin=margin(b = -0.3, unit='cm'),legend.key.height = unit(0.1,"cm")) 
  p <- p + annotate("text",label = label$GF$GF_R2,x = annot_x, y = max(data$value) - annot_y[1] ,
                    size= 4,color = color[1],parse= T) +
    annotate("text",label = label$GF$GF_P,x = annot_x, y = max(data$value) - annot_y[2] ,
             size= 4,color = color[1],parse= T) +
    annotate("text",label = label$SPF$SPF_R2,x = annot_x, y = max(data$value) - annot_y[3],
             size= 4,color = color[2],parse= T) +
    annotate("text",label = label$SPF$SPF_P,x = annot_x, y = max(data$value) -  annot_y[4],
             size= 4,color = color[2],parse= T)
  p <- p + scale_x_continuous(breaks = c(1,2,3,4,5,6,7,8,9),
                                labels = c("L1","L2","L3","L4","L5","L6","L7","L8","L9"))
  return(p)
}

# FigS7 

In [ ]:
DEGBarplot <- function(dat,sample,idx = c("GF","SPF")){  # count # 
  print(idx)
  sample_names <- c(paste0(idx[1]," up-regulated"), paste0(idx[2]," up-regulated"))
  dat <- dat %>%
    dplyr::mutate(group_type = factor(ifelse(group == 1, sample_names[1], sample_names[2]), levels = sample_names)) %>%
    dplyr::mutate(Gene_Number = pvalue * group) %>% dplyr::arrange(pvalue) # add up arrange pvalue
  # 6print(head(dat,20))
  dat <- dat %>% dplyr::group_by(group) %>% dplyr::do(head(., n = 10)) %>% as.data.frame()
  dat$log10Pvalue <- -(log10(dat$pvalue))
  th <- theme(plot.title = element_text(size = 25,vjust = 0.5,hjust = 0.05),
              axis.text.y= element_blank(),
              axis.ticks.y= element_blank(),
              axis.title.x = element_text(size = 30),
              axis.line.x = element_line(size = 1),
              axis.ticks.x = element_line(size = 1),
              axis.text.x = element_text(size = 20),
              panel.background = element_rect(fill = 'white'),
              panel.border = element_rect(color = 'white', fill = NA),
              panel.grid.major.y = element_blank(),
              panel.grid.minor.y = element_blank(),
              panel.grid.major.x = element_line(colour = "grey80", linetype = "dashed"),
              panel.grid.minor.x = element_blank(),
              legend.text=element_text(size=15),
              legend.title= element_text(size=20),
              plot.margin = unit(c(0.5,1,0.5,0.5),"cm"))
  get_wraper <- function(width) {
    function(x) {
      lapply(strwrap(x, width = width, simplify = FALSE), paste, collapse="\n")
    }
  }
  plst <- list()
  colorbar <- data.frame(low = c("#FADDC3","#ADCCF6"),high = c("#983b3a","#537497"))
  for(i in 1:length(idx)){
    pfx <- idx[i]
    print(pfx)
    pdata <- dat %>% filter(mice == pfx) # condition # 
    plst[[pfx]] <- ggplot(pdata, aes(x = reorder(Description, Count,decreasing = T), y = Count)) +  
      geom_bar(aes(fill = log10Pvalue), stat = "identity") + th + 
      labs(x = '', y = '', fill = sample_names[i],title = pfx) + 
      scale_fill_gradient(low = colorbar$low[i] , high = colorbar$high[i]) +  
      coord_flip()   + 
      geom_text(aes(y = ifelse(Gene_Number > 0,0,0), 
                    label= as.vector(Description),
                    hjust = 0),
                size = 5,vjust = 0.5)
  }
  #
  plst[[1]] <- plst[[1]] +  guides(fill=guide_colorbar(title = paste("-log10Pvalue ",sample_names[1],sep = " : ")))
  plst[[2]] <- plst[[2]] + labs(y='Count')
   p <- plst[[1]] / plot_spacer() / plst[[2]] / plot_layout(guides = 'collect',axes ="collect_x",
                                                            heights = c(4,-1.05,4))  & 
     theme(legend.position = 'bottom') &
    plot_annotation(
      title = sample,
      theme = theme(plot.title = element_text(size = 30,vjust = 0.5,hjust = 0.5)))
  
  return(p)
}
GetDegStat <- function(data){
  GetDegNum <- function(data,idx){
    gene = grep(pattern = "^Rp[sl]",x = data$gene,value = T,invert = T) 
    g <- paste(gene,collapse = "/")
    data_n <- data.frame(gene = g, number = length(gene))
    #print(data_n)
    data_n$regulated  <- idx 
    return(data_n)
  }
  glst <- list()
  for(i in unique(data$celltypes)){
    dat <- data %>% filter(celltypes == i )
    up <-  dat %>% filter(group == "up-regulated")
    down <- dat %>% filter(group == "down-regulated")
    if(nrow(up) != 0 && nrow(down) != 0){
       print("1")
       up_n <- GetDegNum(data = up,idx = "up")
       down_n <- GetDegNum(data = down, idx = "down")
       me <- rbind(up_n,down_n)
    } else if(nrow(up) == 0 && nrow(down) !=0){
       print("2")
       down_n <- GetDegNum(data = down, idx = "down")
       me <- down_n
    } else if(nrow(up) !=0 && nrow(down) == 0){
      pirnt("3")
      up_n <- GetDegNum(data = up,idx = "up")
      me <- up_n
    } else if(nrow(up) == 0 && nrow(down) == 0){
      print("4")
      message(paste(i," is not DEG !",sep = ''))
    } else {
      stop("haha")
    }
    me$celltypes <- i
    glst[[i]] <- me
  }
  d <- do.call(rbind,glst)
  return(d)
}
DegBarplot <- function(data,x,y,fill,mycolor,title = NULL){ 
  theme_self <- theme(
    axis.text.y = element_text(size = 25,face = "bold",margin  = margin(0,5,0,0)),
    axis.text.x = element_text(size = 25,face = "bold",margin  = margin(0,0,0,0),angle = 60,hjust = 1,vjust =0.97),
    axis.title.x= element_text(size = 30,face = "bold",margin  = margin(0,0,0,0)),
    axis.title.y= element_text(size = 30,face = "bold",margin  = margin(0,0,0,0)),
    axis.line = element_line(linetype = 1,color= "black",size = 1),
    legend.text = element_text(size = 22,face = "bold"),
    legend.title = element_text(size = 28,face = "bold"),
    panel.grid = element_blank(),
    plot.margin =  unit(c(1,1,1,1),"cm"), # lwm 2023 -7 - 7 
    axis.ticks  = element_line(color = "black",size = 1,lineend = 2),
    plot.title = element_text(hjust = 0.5,size = 30,face = "bold"))
  p <- ggplot(data,aes_string(x = x,y = y,fill = fill)) + 
    geom_bar(stat="identity",width = 0.6) + theme_classic() + labs(x = " " , y = "Number of DEGs",fill = " ") +  
    scale_fill_manual(values=mycolor) + theme_self + 
    theme(legend.position = c(1,1.1),legend.justification = c(1,1)) + ggtitle(title)
  return(p)
}

# FigS8

In [ ]:
##
GetEmbeddings <- function(obj,reduction,clu,ctype = NULL,annot = NULL,idx = NULL){ 
  emb = obj@reductions[[reduction]]@cell.embeddings %>% 
    as.data.frame() %>% cbind(cluster = obj@meta.data[,clu])
  if(!is.null(ctype)){
    emb <- emb %>% cbind(celltype = obj@meta.data[,ctype])
  }
  if(!is.null(annot) & !is.null(ctype)){
    emb <- emb %>% mutate(annot = paste(cluster,celltype,sep = " "))
  }
  if(!is.null(idx)){
    for(i in 1:length(idx)){
      id <-  idx[i]
      print(id)
      emb <- emb %>% cbind(id = obj@meta.data[,id])
      colnames(emb)[length(colnames(emb))] <- id #  last col_names <- id
    }
  }
  return(emb)
}
# 绘图 
DimPlot_custom <- function(emb,dim1,dim2,color,mycolor,label,title=NULL,
                           split = NULL,ellipse = NULL,arrow.num = 1,
                           ncol = 3,xlength = 0.2,ylength = 0.1,
                           arrow.size = 0.1){
  theme_self <- theme(panel.grid = element_blank(), 
                      axis.title = element_blank(),
                      axis.text = element_blank(),
                      axis.ticks = element_blank(),
                      plot.title = element_text(size  = 10,vjust = 0.5, hjust = 0.5),
                      panel.background = element_rect(fill = 'white'), #背景色
                      plot.background=element_rect(fill="white")) +
    theme(legend.title = element_blank(), #去掉legend.title 
          legend.key=element_rect(fill='white'), #
          legend.text = element_text(size=10), #设置legend标签的大小
          legend.key.size=unit(0.5,'cm') )   # 设置legend标签之间的大小
  # theme_dr_self # 
  xlength = 0.2
  ylength = 0.1
  arrow = grid::arrow(length = unit(arrow.size,"inches"), type = "closed")
  theme_dr_self <- theme(axis.line.x.bottom = element_line2(id = 1, xlength = xlength, arrow = arrow), 
                         axis.line.y.left = element_line2(id = 2, ylength = ylength, arrow = arrow),
                         axis.title.x = element_text(hjust = 0.1),
                         axis.title.y = element_text(hjust = 0.05))
  
  dimplot <- function(emb,dim1,dim2,color,center,mycolor,label){
    p1 <- ggplot(emb,aes_string(x= dim1 , y = dim2 ,color = color)) +  
      geom_point_rast(size = 0.001 , alpha = 0.8 )  + 
      #geom_point(size = 0.001,alpha = 0.8) + 
      scale_color_manual(values = mycolor) + 
      theme_self + guides(color = guide_legend(override.aes = list(size=5))) +
      geom_text_repel(aes_string(label = label), 
                      fontface="bold",data = center, 
                      point.padding=unit(0.1, "lines"),color = "black")
    return(p1)
  }
  # GetTypeCenter # 
  GetTypeCenter <- function(emb,idx,dim1,dim2){
    col_names <- c(idx,dim1,dim2)
    emb <- emb[,col_names] # select # 
    center <- emb %>% dplyr::group_by(.data[[idx]]) %>% 
      dplyr::summarise(e1 = median(.data[[dim1]]),
                e2 = median(.data[[dim2]]))  %>% as.data.frame()
    colnames(center) <- col_names
    return(center)
  }
  center <- GetTypeCenter(emb =emb,idx = label,dim1 = dim1, dim2 = dim2) 
  
  if(!is.null(split)){
    plst <- list()
    for(i in unique(emb[,split])){
      print(i)
      em <- emb %>% filter(.data[[split]] == i)
      #cen <- GetTypeCenter(emb =em,idx = label ,dim1,dim2) 
      print(head(center))
      plst[[i]] <- dimplot(emb = em,dim1 = dim1,dim2 = dim2,center = center, # all center same 
                           mycolor = mycolor,color = color,label = label) + ggtitle(i)
      if(length(unique(em[,color])) != length(unique(emb[,color]))){
        # legend num may be lack : check and remove
        plst[[i]] <- plst[[i]] + theme(legend.position = "none") 
      }
    }
    # plst[[arrow.num]] <- plst[[arrow.num]] + theme_dr_self
    p <- wrap_plots(plst,ncol = ncol) + plot_layout(guides = 'collect')
    p <- p & theme_dr_self 
  } else {
    # center <- GetTypeCenter(emb =emb,idx = label,dim1,dim2)
    p <- dimplot(emb = emb,dim1 = dim1,dim2 = dim2,center = center,
                 mycolor = mycolor,color = color,label = label) + theme_dr_self + 
      ggtitle(title)
  }
  if(!is.null(ellipse)){
    p2 <- p + stat_ellipse(aes(fill = color),level = 0.95,linetype = 2,
                           show.legend = F,geom = 'polygon',alpha = 0.1) 
  } else {
    p2 <- p
  }
  return(p2)
}

# FigS12 

In [ ]:
ToCluCellType <-function(object){
    meta <- object@meta.data[,c('clusters','celltypes')]
    meta$celltypes <- as.vector(meta$celltypes)
    meta$clusters <- as.numeric(as.vector(meta$clusters))
    meta <- meta %>% arrange(clusters) %>% mutate(clu_type = paste(clusters,celltypes,sep = " "))
    clu_type <- unique(meta$clu_type)
    return(clu_type)
}
IntestineUmap <- function(object,colorlist){
    clu_num <- length(unique(object@meta.data$clusters))
    cat("number of cluster : ",clu_num , "\n")
    object@meta.data$clusters <- factor(object@meta.data$clusters,levels = 0:clu_num)
    ct_labels <- colorlist[["labels"]]
    if(clu_num <= length(colorlist[["type"]])){
            ct_type <- rev(colorlist[["type"]][1:length(unique(ct_labels))])
    } else {
        stop("number of color is not enough")
    }
    print(ct_labels)
    print(ct_type)
    theme_arrow <- theme_dr(xlength = 0.2,ylength = 0.2,arrow = grid::arrow(length = unit(0.1, "inches"),
                                                                         ends = 'last', type = "closed"))
    theme_custom <- theme(plot.title=element_blank(),
                 axis.title = element_text(size =10,color ="black"),#修改坐标轴的文字描述
                 axis.text = element_blank(),
                 panel.grid.minor.y = element_blank(),
                 panel.grid.minor.x = element_blank(),
                  panel.grid=element_blank(), # 去网格线
                 legend.text = element_text(size=14),#修改legend字体
                 legend.title= element_text(size= 18),
                 axis.text.x = element_blank(),
                 plot.margin = unit(c(0.5,0.5,0.5,0.5),"lines"))   
    CtUmap <- DimPlot(object, label = T,label.size=6, pt.size = 3,reduction ="tsne",group.by = "clusters",raster=TRUE)+ 
              theme_arrow + theme_custom +  scale_color_manual(name = "", 
                     values = ct_type,
                     labels = ct_labels,
                      )+coord_fixed()

    TsUmap <- DimPlot(object, label = F,label.size=6, pt.size = 3,reduction ="tsne", group.by="tissue",raster=TRUE) + 
              theme_arrow + theme_custom +
                scale_color_discrete(name = "", 
                     type =c("#3E9FB3","gray50","#C075A6"),
                     labels = rev(c("Cecum","Colon","Ileum"))) +coord_fixed()
    MtUmap <- FeaturePlot(object, pt.size =1.5,features=c("percent.mt"),raster=TRUE,
                          reduction ="tsne",cols=c("lightgrey", "darkblue"))+theme_arrow + theme_custom + coord_fixed()
    wholeUmap <- CtUmap + TsUmap + MtUmap 
    return(wholeUmap)
}
date()

# FigS15 

In [ ]:
ErrorLinePlot <- function(data,y_title,colorplate = c("#F8766D","#00BFC4")){
  theme_self <- theme(
    axis.text.y = element_text(size = 15,face = "bold",margin  = margin(0,5,0,0)),
    axis.text.x = element_text(size = 15,face = "bold",margin  = margin(5,0,0,0)),
    axis.title.x= element_text(size = 20,face = "bold",margin  = margin(10,0,0,0)),
    axis.title.y= element_text(size = 20,face = "bold",margin  = margin(0,10,0,0)),
    axis.line = element_line(linetype = 1,color= "black",size = 1),
    legend.text = element_text(size = 12,face = "bold"),
    legend.title = element_text(size = 15,face = "bold"),
    #panel.grid = element_blank(),
    plot.margin =  unit(c(0.3,0.3,0.3,0.3),"cm"), # lwm 2023 -7 - 7 
    axis.ticks  = element_line(color = "black",size = 1,lineend = 2),
    plot.title = element_text(hjust = 0.5,size = 30,face = "bold"))
  p <- ggplot(data,aes(x = Layer,y = value,color = Mice,group = Mice)) + 
    geom_errorbar(aes(ymin = value - se,ymax = value + se),width = 0.2) + geom_line()  + 
    geom_point(size =3,shape = 20)  + labs(y = y_title )  + 
    scale_color_manual(values = c(GF= colorplate[1],SPF = colorplate[2]))  + theme_bw() + theme_self 
    #theme(legend.position = c(1,1),legend.justification = c(1,1),
    #      legend.margin=margin(b = -0.4, unit='cm'),legend.key.height = unit(0.1,"cm")) # 
  return(p)
}

# FigS16

In [ ]:
AddSignif <- function(stat,var,signif.data,thread = NULL){
    if(!is.null(thread)){
        cut <- thread
    } else { 
        cut = c(0, 1e-04, 0.001, 0.01, 0.05, 1)
    }
    signif.data <- signif.data %>% add_significance(cutpoints = cut)
    dat.list <- list()
    for(i in 1:length(unique(stat[,var]))){
        print(i)
        sp <- unique(stat[,var])[i]
        dat <- stat[stat[,var] == sp,c("celltypes",var)]
        colnames(dat) <- c("celltypes",paste("group",i,sep = ""))
        print(head(dat))
        dat.list[[i]] <- dat
    }
    group <- do.call(left_join,c(dat.list,by = "celltypes"))                                                                      
    signif.data <- left_join(group,signif.data)
    signif.data$xmin = 1
    signif.data$xmax = 2
    signif.data <- signif.data[order(signif.data$yposition),]
    signif.data$celltypes <- factor(signif.data$celltypes,levels = signif.data$celltypes)
    return(signif.data)
}

SfBarplotPlot <- function(plot.data,signif.data,mycolor,bar_width = 0.7,facet = FALSE,lg.pos = "top"){
    theme_self <- theme(plot.title = element_text(hjust = 0.5,size = 20,face = "bold"),
            axis.text.y = element_text(size = 16,face = "bold"),
            axis.text.x = element_text(size = 16,angle = 45,hjust = 1,face = "bold"),
            axis.title.x= element_text(size = 20,face = "bold",margin = margin(0,10,0,0)),
            axis.title.y= element_text(size = 20,face = "bold"),
             axis.line = element_line(linetype = 1,color= "black",size = 1),
            legend.text = element_text(size = 16,face = "bold"),
            legend.title = element_text(size = 20,face = "bold"),
            plot.margin =  unit(c(1,2,1,2),"cm"))
    plot.data$celltypes <- factor(plot.data$celltypes,levels = signif.data$celltypes)
    num <- length(unique(signif.data$celltypes))
    if(is.null(mycolor)){
        mycolor = colorRampPalette(brewer.pal(9, "Set1"))(num)
    } 
    if(facet == TRUE){ # no check !!! 
        p <- ggplot(plot.data,aes(x = mice,y = percentage,fill = celltypes)) +
                geom_bar(stat = "identity",position = "dodge",width = bar_width,colour = "black") + 
                facet_grid(. ~ celltypes) + 
                theme_classic() + ylab("The percentage of celltypes") +xlab("") +
                theme(legend.position = 'none')+ scale_fill_manual(values = mycolor) 
        p2 <- p + stat_pvalue_manual(data = signif.data,label = "pvalue.signif",y.position = "yposition",size = 6) + 
                theme(strip.background = element_blank(),strip.text =  element_text(size = 15))  + theme_self
    } else {
        print(num)
        s <- seq(1,num,1)
        signif.data$xmin <- s - bar_width/4
        signif.data$xmax <- s + bar_width/4
        p2 <- ggbarplot(plot.data,x ="celltypes",y="percentage",fill = "mice",color= "mice", 
                        palette = mycolor,position = position_dodge(bar_width)) + 
                        stat_pvalue_manual(data = signif.data,label = "pvalue.signif",
                                                       y.position = "yposition",size = 6) + 
        ylab("The percentage of celltypes") + theme_self + theme(legend.position= lg.pos)
    }
    return(p2)
 }
DEGBarplot <- function(dat,sample){ # add OPname to output pdf 2022-12-9
  sample_names <- c(paste0('GF'," up-regulated"), paste0('SPF'," up-regulated"))
  dat <- dat %>%
    dplyr::mutate(group_type = factor(ifelse(group == 1, sample_names[1], sample_names[2]), levels = sample_names)) %>%
        dplyr::mutate(Gene_Number = Count * group)
  dat <- dat %>% dplyr::group_by(group) %>% dplyr::do(head(., n = 10)) %>% as.data.frame()
  th <- theme(axis.text.y= element_blank(),
              axis.ticks.y= element_blank(),
              axis.title.x = element_text(size = 30),
              axis.line.x = element_line(size = 1),
              axis.ticks.x = element_line(size = 1),
              axis.text.x = element_text(size = 20),
              panel.background = element_rect(fill = 'white'),
              panel.border = element_rect(color = 'white', fill = NA),
              panel.grid.major.y = element_blank(),
              panel.grid.minor.y = element_blank(),
              panel.grid.major.x = element_line(colour = "grey80", linetype = "dashed"),
              panel.grid.minor.x = element_blank(),
              legend.text=element_text(size=20),
              legend.title= element_text(size=25),
              plot.margin = unit(c(0.5,0,0.5,0),"cm"))
  get_wraper <- function(width) {
    function(x) {
      lapply(strwrap(x, width = width, simplify = FALSE), paste, collapse="\n")
    }
  }
  print(head(dat,1))
  p <- ggplot(dat, aes(x = reorder(Description, Gene_Number), y = Gene_Number)) +
    geom_bar(aes(fill = group_type), stat = "identity") + th + 
    labs(x = '', y = 'Gene Number', fill = sample) + 
    scale_fill_manual(values=c("#983b3a","#537497")) + 
    coord_flip() + theme(legend.position = 'bottom') + guides(fill = guide_legend(ncol = 2))+
    geom_text(aes(y = ifelse(Gene_Number > 0,0,0), # -0.5,0.5
                  label= as.vector(lapply(strwrap(reorder(Description, Gene_Number), 
                           width = 65, simplify = FALSE), paste, collapse="\n")),
                  hjust = ifelse(Gene_Number > 0,1,0)),size = 5,vjust = 0.5)
  return(p)
}